# Lesson 05 Lab — Feeding the SM and Tensor Cores

**Puzzle:** Why can a Tensor Core capable GPU underperform when a matrix shape is only slightly awkward?

This notebook retains one complete RTX 5090 execution.


## Why this matters

Tensor Cores are execution units inside an SM, not autonomous matrix servers. Instructions must be scheduled, operands must be fetched from register banks and collectors, and tiles must match a supported dtype and layout. Memory, instruction issue, dependency tracking, register pressure, and tile geometry can all prevent the arithmetic pipeline from staying full.


## 0. Predict before running

1. Predict which shape the library will execute more efficiently.
2. List the path from L2 response to matrix operands.
3. Name the evidence required to assert Tensor Core dispatch.

For each prediction, write the observation that would disprove it.


## 1. Theory and mechanism

The experiment compares a well-aligned BF16 GEMM with an awkward shape using the same approximate FLOP scale. It reports time and achieved throughput, then reads the GPU compute capability. A faster aligned case is evidence about these two library-dispatched shapes, not proof that one specific Tensor Core instruction executed; that claim would require a kernel or profiler trace.

- An SM combines scheduling, storage, load/store, scalar/vector, and matrix resources.
- Tensor Core throughput depends on a supported instruction and a fed pipeline.
- Shape alignment is an empirical library contract, not a universal multiple copied from a blog.


## 2. Trace the mechanism

### Mechanism map

```mermaid
flowchart LR
  A["warp scheduler"] --> B["scoreboard"]
  B --> C["register banks"]
  C --> D["operand collector"]
  D --> E["Tensor Core MMA"]
  E --> F["accumulator"]
```


## 3. Inspect the visual boundary

This lesson is driven by a Mermaid mechanism map and executable measurements.


## 4. Inspect the execution environment

The next cell asserts CUDA, records GPU/PyTorch/CUDA identity, fixes the seed, and defines the common event-timing helpers.


In [1]:
LESSON_NO = 5
LESSON_TITLE = 'Feeding the SM and Tensor Cores'

from pathlib import Path
from collections import Counter, deque
import json, math, platform, statistics, sys, time

import torch
import torch.nn.functional as F

assert torch.cuda.is_available(), "Chapter 04 retained runs require a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260813 + LESSON_NO
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

major, minor = torch.cuda.get_device_capability(0)
props = torch.cuda.get_device_properties(0)
ENV = {
    "gpu": torch.cuda.get_device_name(0),
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    pos = (len(ordered) - 1) * q
    lo, hi = math.floor(pos), math.ceil(pos)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def cuda_samples(fn, warmup=5, repeats=20):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    samples = []
    for _ in range(repeats):
        start = torch.cuda.Event(enable_timing=True)
        stop = torch.cuda.Event(enable_timing=True)
        start.record()
        fn()
        stop.record()
        stop.synchronize()
        samples.append(float(start.elapsed_time(stop)))
    return samples

def summary(samples):
    return {
        "median_ms": statistics.median(samples),
        "p95_ms": percentile(samples, 0.95),
        "samples_ms": samples,
    }


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "seed": 20260818
}


## 5. Freeze the experiment

| Role | Frozen value |
|---|---|
| Baseline | square dimensions aligned to common library tiles |
| Candidate | nearby awkward M/N/K dimensions |
| Held constant | dtype, GPU, timing, warm-up, and approximate FLOP count |
| Measurements | median latency, achieved TFLOP/s, and throughput ratio |
| Evidence | `pytorch-gpu` |

**Experiment:** Time aligned and awkward BF16 GEMMs with similar arithmetic scale.


## 6. Inspect the code

Both candidates call `torch.mm`, so the installed PyTorch/cuBLAS stack selects tactics. The code synchronizes with events and validates output shapes; it does not label internal instructions without a trace.

Do not run until the code matches the frozen table.


In [2]:
dtype = torch.bfloat16

def make_case(m, n, k):
    a = torch.randn((m, k), device=DEVICE, dtype=dtype)
    b = torch.randn((k, n), device=DEVICE, dtype=dtype)
    out = torch.empty((m, n), device=DEVICE, dtype=dtype)
    samples = cuda_samples(lambda: torch.mm(a, b, out=out), repeats=20)
    median = statistics.median(samples)
    return {"shape": [m, n, k], "median_ms": median,
            "tflops": (2 * m * n * k) / (median / 1e3) / 1e12,
            "samples_ms": samples, "checksum": float(out.float().mean().item())}

aligned = make_case(2048, 2048, 2048)
awkward = make_case(2039, 2053, 2041)
metrics = {
    "compute_capability": ENV["compute_capability"],
    "aligned": aligned,
    "awkward": awkward,
    "aligned_median_ms": aligned["median_ms"],
    "awkward_median_ms": awkward["median_ms"],
    "aligned_tflops": aligned["tflops"],
    "awkward_tflops": awkward["tflops"],
    "throughput_ratio": aligned["tflops"] / awkward["tflops"],
}
analysis = (
    f"The aligned and awkward BF16 shapes reached {aligned['tflops']:.1f} and "
    f"{awkward['tflops']:.1f} TFLOP/s. This establishes a library-shape effect on this stack; "
    "it does not identify internal instructions without a profiler trace."
)
print(json.dumps(metrics, indent=2))


{
  "compute_capability": "12.0",
  "aligned": {
    "shape": [
      2048,
      2048,
      2048
    ],
    "median_ms": 0.10390400141477585,
    "tflops": 165.3436725253673,
    "samples_ms": [
      0.11264000087976456,
      0.106175996363163,
      0.10540799796581268,
      0.1032319962978363,
      0.1043199971318245,
      0.10291200131177902,
      0.10460799932479858,
      0.10300800204277039,
      0.10400000214576721,
      0.1037760004401207,
      0.10387200117111206,
      0.10393600165843964,
      0.10387200117111206,
      0.10419200360774994,
      0.10345599800348282,
      0.1032319962978363,
      0.10400000214576721,
      0.10374400019645691,
      0.1032319962978363,
      0.10400000214576721
    ],
    "checksum": 0.042515262961387634
  },
  "awkward": {
    "shape": [
      2039,
      2053,
      2041
    ],
    "median_ms": 0.17475200444459915,
    "tflops": 97.7815707940402,
    "samples_ms": [
      0.1847359985113144,
      0.17759999632835388,
      0

## 7. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Aligned median | 0.104 ms |
| Awkward median | 0.175 ms |
| Aligned throughput | 165.3437 |
| Awkward throughput | 97.7816 |
| Aligned/awkward ratio | 1.691x |


## 8. Explain rather than overclaim

The aligned and awkward BF16 shapes reached 165.3 and 97.8 TFLOP/s. This establishes a library-shape effect on this stack; it does not identify internal instructions without a profiler trace.

**Evidence boundary:** CUDA work executed through PyTorch. It does not identify an internal instruction, cache event, or proprietary hardware block without additional profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, metrics, analysis, evidence label, and bounded conclusion, then prints the exact JSON.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 5, "title": 'Feeding the SM and Tensor Cores', "environment": ENV,
    "evidence_label": 'pytorch-gpu', "metrics": metrics,
    "analysis": analysis, "conclusion": 'Treat alignment as a measured performance variable and preserve the full shape/dtype/backend identity with every result.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 5,
  "title": "Feeding the SM and Tensor Cores",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "seed": 20260818
  },
  "evidence_label": "pytorch-gpu",
  "metrics": {
    "compute_capability": "12.0",
    "aligned": {
      "shape": [
        2048,
        2048,
        2048
      ],
      "median_ms": 0.10390400141477585,
      "tflops": 165.3436725253673,
      "samples_ms": [
        0.11264000087976456,
        0.106175996363163,
        0.10540799796581268,
        0.1032319962978363,
        0.1043199971318245,
        0.10291200131177902,
        0.10460799932479858,
        0.10300800204277039,
        0.10400000214576721,
        0.1037760004401207,
        0.10387200117111206,
        0.10393600165843964,
        0.10387200117111206,
        0.10419200360774994,
        0.10345599800348282,
        0.1032319962978363,
        0.104000

## 10. Make the decision

> Treat alignment as a measured performance variable and preserve the full shape/dtype/backend identity with every result.

**Failure analysis:** Library autotuning, clocks, workspace, and architecture can select different kernels. Awkward does not always mean slower, especially when the total work is smaller.


## 11. Extend the evidence

Capture an Nsight Compute instruction mix and sweep each dimension independently around several tile boundaries.

See [`README.md`](README.md) for the full explanation and references.
